In [1]:
import numpy as np
import pandas as pd

#### Step 1: Indexing - A) Document Injection 
Loading PDF documents using LangChain's PDFPlumberLoader.

validation of the Documents:
1. file size (Docs level)
2. MIME Type (Docs level)
3. PDF Corruption check (Docs level)
4. Readibility score (Text level)
5. OCR Quality (text level)

In [2]:
pip install -qU langchain-community pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Loading the PDF document which contains tables and text
# Note: PyPDFLoader may not handle tables well, so we are using PDFPlumberLoader
from langchain_community.document_loaders import PDFPlumberLoader

pdf_path = "D:\Shridhar\Data Science && Data Analytics\GenAI\RAG Based LLM model to know about Indian Cricket\Indian_Cricket_Report.pdf"
loader = PDFPlumberLoader(pdf_path)
documents = loader.load()     # need watch, how the documents are loaded and its number

print(f"Loaded {len(documents)} document(s) from the PDF.")
print(f"First document preview: {documents[2].page_content[:500]}...")

Loaded 6 document(s) from the PDF.
First document preview: 3. Domestic Cricket Structure
3.1 Major Domestic Tournaments
Tournament Format Established
Ranji Trophy First-class 1934
Duleep Trophy First-class 1961
Irani Cup First-class 1959
Vijay Hazare Trophy List A 2002
Syed Mushtaq Ali Trophy T20 2006
Deodhar Trophy List A 1973
3.2 Indian Premier League (IPL)
The Indian Premier League, established in 2008, has revolutionized cricket globally. It's the
most-attended cricket league in the world and has the highest brand value among cricket leagues. The
IP...


In [5]:
import os
# Function to validate file size (for controlling costs in cloud services)
size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
def validate_file_size(file_path, max_size_mb=50):
    if size_mb > max_size_mb:
        raise ValueError(f"File exceeds size limit: {size_mb:.2f} MB")
    return True

validate_file_size(pdf_path)
print("File size (in MB) of the current Document for checking, is within the limits:", size_mb)


File size (in MB) of the current Document for checking, is within the limits: 0.010663986206054688


In [57]:
pip install python-magic-bin

Note: you may need to restart the kernel to use updated packages.


In [6]:
import magic
# Function to validate MIME type : to ensure the data source is as expected during file Uploads

def validate_mime_type(file_path):
    mime = magic.from_file(file_path, mime=True)
    if mime != "application/pdf":
        raise ValueError(f"Invalid MIME type: {mime}")
    return True

validate_mime_type(pdf_path)

True

In [7]:
pip install PyMuPDF


Note: you may need to restart the kernel to use updated packages.


In [7]:
# function to check PDF corruption: 
import fitz  # PyMuPDF

def validate_pdf_integrity(file_path):
    try:
        doc = fitz.open(file_path)
        if doc.page_count == 0:
            raise ValueError("PDF has zero pages")
        doc.close()
    except Exception as e:
        raise ValueError(f"Corrupted PDF: {str(e)}")
    return True
validate_pdf_integrity(pdf_path)

True

In [9]:
pip install textstat

Note: you may need to restart the kernel to use updated packages.


In [8]:
import textstat
# The Ideal Readibility Score can be set as per the use-case requirements. Here for a ministry related document, the readibility score comes under difficult level
# Difficult level readibilty score is between 30-50. Here we set the minimum score as 18 to allow for some flexibility.
def validate_readability(text, min_score=18):
    score = textstat.flesch_reading_ease(text)
    if score < min_score:
        raise ValueError(f"Low readability score: {score}")
    return score

print("Readibility Score for this Document:", validate_readability(documents[0].page_content))

Readibility Score for this Document: 38.01822368421057


In [9]:
#OCR Quality Validation: Check the ratio of alphabetic characters to total characters
def validate_ocr_quality(text, min_alpha_ratio=0.7):
    if not text:
        raise ValueError("Empty text: possible OCR failure")
    
    # Count alphabetic characters and spaces (common in readable text)
    alpha_space_count = sum(1 for c in text if c.isalpha() or c.isspace())
    total_count = len(text)
    ratio = alpha_space_count / total_count
    
    if ratio < min_alpha_ratio:
        raise ValueError(f"Poor OCR quality: alpha/space ratio {ratio:.2f} below threshold {min_alpha_ratio}")
    
    return ratio

# Calculating OCR quality for total documents
docs_length = len(documents)
total_ratios = 0
for doc in documents:
    total_ratios += validate_ocr_quality(doc.page_content)

print("Average OCR Quality Ratio:", total_ratios / docs_length if docs_length > 0 else 0)

Average OCR Quality Ratio: 0.8850177118038776


##### Removing Table from Text 

why? : Tables as text lose structure—embeddings might not capture numerical relationships or columns well, leading to noisy or irrelevant retrievals and have data duplicy issues (this is one of the limitations of the vecDB).

Why Not?: This allows semantic search over table content using vector similarity and Useful if queries are conversational and don't require exact tabular operations (e.g., sums, filters).

Conclusion: Summary of tables (Both Inside and Outside Document file), we will add in the metadata (Json) along with document summary and then we store metadata + Text (their Embeddings) in vector DB. By this we can able to solve issues such as if our queries mix text and tables (e.g., "Explain the tariff regulations and show CEA data of Discom"). It can enhance coverage but risks inconsistency (e.g., vecDB might retrieve outdated table text while SQL has the latest).


In [12]:
pip install langchain-core

Note: you may need to restart the kernel to use updated packages.


In [10]:
extracted_text_documents = documents
# # Removing tables from document and   text for clean chunking (vector DB) - Improved approach using bounding boxes
# import pdfplumber
# from langchain_core.documents import Document
# import hashlib
# from datetime import datetime

# extracted_text_documents = []
# with pdfplumber.open(pdf_path) as pdf:
#     for page_num, page in enumerate(pdf.pages, start=1):
#         # Find all tables on the page and collect their bounding boxes
#         tables = page.find_tables()
#         table_bboxes = [table.bbox for table in tables]  # List of (x0, y0, x1, y1) tuples
        
#         # Get all characters on the page with their bounding boxes
#         chars = page.chars
        
#         # Filter out characters that are inside any table bounding box
#         filtered_chars = []
#         for char in chars:
#             char_bbox = (char['x0'], char['top'], char['x1'], char['bottom'])  # Note: 'top' and 'bottom' for y-coords
#             is_in_table = any(
#                 char_bbox[0] >= table_bbox[0] and char_bbox[1] >= table_bbox[1] and
#                 char_bbox[2] <= table_bbox[2] and char_bbox[3] <= table_bbox[3]
#                 for table_bbox in table_bboxes
#             )
#             if not is_in_table:
#                 filtered_chars.append(char)
        
#         # Reconstruct text from filtered characters
#         cleaned_text = ''.join(char['text'] for char in filtered_chars)
        
#         # Skip pages with no remaining text
#         if not cleaned_text.strip():
#             continue
        
#         keywords = ['tariff', 'CERC', 'amendment']  # Add relevant keywords with your reavancy.
#         extracted_text_documents.append(Document(
#             page_content=cleaned_text,
#             metadata={
#                 # Basic Info
#                 'source': pdf_path,
#                 'page': page_num,
#                 'total_pages':len(extracted_text_documents),
                
#                 # Document Classification    | we have to add sepatrate logic for other documents as well (More than one pdfs)     
#                 'document_type': 'regulatory',
#                 'document_title': 'CERC 2024-2029 Tariff Regulations',
#                 'document_category': 'amendment',
                
#                 # Quality Metrics
#                 'char_count': len(cleaned_text),
#                 'has_tables': len(tables) > 0,
#                 'table_count': len(tables),
                
#                 # Searchability Enhancement
#                 'section': 'regulation',  # Can be extracted from headings
#                 'keywords': ', '.join(keywords),  # Add relevant keywords
                
#                 # Versioning & Tracking
#                 'extraction_date': datetime.now().isoformat(),
#                 'content_hash': hashlib.md5(cleaned_text.encode()).hexdigest(),
#             }
#         ))

# print(f"Created {len(extracted_text_documents)} cleaned documents.")
# print(f"First cleaned document preview: {extracted_text_documents[0].page_content[:500]}...")

In [11]:
# just checking the OCR quality of cleaned documents(after table removal)
docs_length2 = len(extracted_text_documents)
total_ratios2 = 0
for doc in extracted_text_documents:
    total_ratios2 += validate_ocr_quality(doc.page_content)

print("Average OCR Quality Ratio:", total_ratios2 / docs_length2 if docs_length2 > 0 else 0)

Average OCR Quality Ratio: 0.8850177118038776


#### Step 1: Indexing - B) text chunking 
Spliting the loading text into manageable chunks using RecursiveCharacterTextSplitter.

In [12]:
# Text Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Size of each chunk in characters
    chunk_overlap=250  # Overlap between chunks
)
chunks = text_splitter.split_documents(extracted_text_documents)

for i, chunk in enumerate(chunks):
    chunk.metadata.update({
        'chunk_id': i,
        'chunk_size': len(chunk.page_content),
        'readability_score': textstat.flesch_reading_ease(chunk.page_content),
        # You can add semantic tags here based on content analysis
    })

print(f"Total chunks created: {len(chunks)}")
print("Readibility Score for this chunk:", validate_readability(chunks[0].page_content))
print(f"length of first chunk:", len(chunks[0].page_content))    # chunk size is valid length of each chunk should less than or equal to chunk_size
print(f"First chunk preview: {chunks[0].page_content}...")

Total chunks created: 10
Readibility Score for this chunk: 41.614223602484486
length of first chunk: 1000
First chunk preview: Comprehensive Report on Indian
Cricket
1. Introduction and Historical Overview
Indian cricket represents one of the most significant cultural and sporting phenomena in modern India.
The Board of Control for Cricket in India (BCCI), established in 1928, governs cricket in India and is the
richest cricket board in the world. India's journey in international cricket began in 1932 with its first Test
match against England at Lord's Cricket Ground.
Year Historical Event
1932 India played its first Test match against England
1952 First Test victory against England in Chennai
1971 First series wins in England and West Indies
1983 First Cricket World Cup victory under Kapil Dev
2007 ICC T20 World Cup victory
2011 Second ODI World Cup victory (host nation)
2013 Champions Trophy victory under MS Dhoni
2021 Historic Test series win in Australia
1.1 Founding Figures of Ind

In [15]:
pip install langchain-huggingface


Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


(A). Preprocessing text to store it in VectorDB

Important Points:
1. Here i will store metadata + text Embeddings along with table data (for checking whether model is able to give response including queries which are related to table)

Tool: ChromaDB (It stores both metadata and embedding internally)

issues:

solutions:

In [19]:
pip install langchain-community langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [13]:
# For text embedding, currently we are using Hugging Face Model (for example: "sentence-transformers/all-MiniLM-L6-v2" )
# we can use OpenAI embedding models as well which requires API key and may incur costs (For better results of query search optimization).
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")  # Persists to disk for reuse
print("Vector store created with ChromaDB.")

Vector store created with ChromaDB.


In [19]:
pip install langchain transformers

Note: you may need to restart the kernel to use updated packages.


In [14]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# Load a pre-trained LLM pipeline (adjust model for your needs)
llm_pipeline = pipeline(
    "text2text-generation",  # For generative tasks like QA
    model="google/flan-t5-small",  # Free, small model; replace with larger if needed
    max_length=512,
    temperature=0.7
)
llm = HuggingFacePipeline(pipeline=llm_pipeline)

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
C:\Users\hp\AppData\Local\Temp\ipykernel_9904\2817109157.py:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=llm_pipeline)


In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define a prompt template for the RAG chain : As of now we are using a simple template
template = """You are a helpful assistant answering questions about Indian Cricket.
Answer the question based only on the provided context. If the answer is not in the context, say "I don't have information about this in the provided documents."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

# the RAG chain using LCEL
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 5,
        # Number of relevant documents to retrieve you can adjust filter as per your use-case
    }
)

# Chain: retrieve documents, format them, and pass to LLM
def format_docs(docs):
    """Format retrieved documents into a string."""
    return "\n\n".join(doc.page_content for doc in docs)


# Simpler LCEL version (recommended) : modify this for my use-case
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [78]:
# Test the chain with advanced formatting
import textwrap
from IPython.display import display, Markdown, HTML

query = input("Enter your question: ")
response = rag_chain.invoke(query)

# Format and display the response
print("="*80)
print("QUESTION:")
print("="*80)
wrapped_query = textwrap.fill(query, width=80)
print(wrapped_query)

print("\n" + "="*80)
print("GENERATED RESPONSE:")
print("="*80)

# Wrap the response text for better readability
wrapped_response = textwrap.fill(response, width=80)
print(wrapped_response)

# Display as Markdown for even better formatting (optional)
display(Markdown(f"### 📄 Response:\n\n{response}"))

# Retrieve and display source documents with rich metadata
docs = retriever.invoke(query)
print("\n" + "="*80)
print(f"SOURCES ({len(docs)} documents retrieved):")
print("="*80)

for idx, doc in enumerate(docs, 1):
    print(f"\n📌 Source {idx}:")
    print(f"   • Page: {doc.metadata.get('page', 'N/A')}")
    print(f"   • Document: {doc.metadata.get('document_title', 'N/A')}")
    print(f"   • Chunk ID: {doc.metadata.get('chunk_id', 'N/A')}")
    print(f"   • Has Tables: {doc.metadata.get('has_tables', 'N/A')}")
    print(f"   • Content Preview:")
    
    # Wrap the content preview for readability
    preview = doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content
    wrapped_preview = textwrap.fill(preview, width=75, initial_indent="     ", subsequent_indent="     ")
    print(wrapped_preview)
    print("-" * 80)


QUESTION:
tell me about ipl

GENERATED RESPONSE:
The Indian Premier League, established in 2008, has revolutionized cricket
globally. It's the most-attended cricket league in the world and has the highest
brand value among cricket leagues. TheIPL has produced numerous international
stars and changed the financial landscape of cricket.eee3.3 IPL Financial
ImpactMetricValueBrand Value (2023)$10.7 billion 3. Domestic Cricket
Structure3.1 Major Domestic Tournaments Tournament Format Established Ranji
Trophy First-class 1934 Duleep Trophy First-class 1961 Irani Cup First-class
1959 Vijay Hazare Trophy List A 2002 Syed Mushtaq Ali Trophy T20 2006 Deodhar
Trophy List A 1973 3.2 Indian Premier League (IPL) The Indian Premier League,
established in 2008, has revolutionized cricket globally. It's the most-attended
cricket league in the world and has the highest brand value among cricket
leagues. TheIPL has produced numerous international stars and changed the
financial landscape of cricket.


### 📄 Response:

The Indian Premier League, established in 2008, has revolutionized cricket globally. It's the most-attended cricket league in the world and has the highest brand value among cricket leagues. TheIPL has produced numerous international stars and changed the financial landscape of cricket.eee3.3 IPL Financial ImpactMetricValueBrand Value (2023)$10.7 billion 3. Domestic Cricket Structure3.1 Major Domestic Tournaments Tournament Format Established Ranji Trophy First-class 1934 Duleep Trophy First-class 1961 Irani Cup First-class 1959 Vijay Hazare Trophy List A 2002 Syed Mushtaq Ali Trophy T20 2006 Deodhar Trophy List A 1973 3.2 Indian Premier League (IPL) The Indian Premier League, established in 2008, has revolutionized cricket globally. It's the most-attended cricket league in the world and has the highest brand value among cricket leagues. TheIPL has produced numerous international stars and changed the financial landscape of cricket.


SOURCES (5 documents retrieved):

📌 Source 1:
   • Page: 3
   • Document: CERC 2024-2029 Tariff Regulations
   • Chunk ID: 2
   • Has Tables: True
   • Content Preview:
     3. Domestic Cricket Structure3.1 Major Domestic Tournaments3.2 Indian
     Premier League (IPL)The Indian Premier League, established in 2008,
     has revolutionized cricket globally. It's themost-attended cricket
     league in the world and has the highest brand value among cricket
     leagues. TheIPL has produced ...
--------------------------------------------------------------------------------

📌 Source 2:
   • Page: 2
   • Document: N/A
   • Chunk ID: 4
   • Has Tables: N/A
   • Content Preview:
     3. Domestic Cricket Structure 3.1 Major Domestic Tournaments
     Tournament Format Established Ranji Trophy First-class 1934 Duleep
     Trophy First-class 1961 Irani Cup First-class 1959 Vijay Hazare Trophy
     List A 2002 Syed Mushtaq Ali Trophy T20 2006 Deodhar Trophy List A
     1973 3.2 Indian Premier 

(B). Making another DB to Store the Tabular Data

Purpose:
1. if we store it in vecDB its embedding will store as the value of a text.
2. we also need CEA (which is purely table formate) data for this model, so we also need to store it somewhere.

issues:

solutions:


In [ ]:
# Extract tables from the PDF
import pdfplumber
tables = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_tables = page.extract_tables()
        for table in page_tables:
            # Convert to DataFrame for easier handling
            df = pd.DataFrame(table[1:], columns=table[0])  # Assuming first row is headers
            tables.append(df)

print(f"Extracted {len(tables)} tables.")
# Example: Print first table
if tables:
    print(tables[0])
else:
    print("No tables found.")
    
    
# confirm and complete, how it will be processed  to store in SQL database or if needed to store in vector DB (by chunking)
# after extracing tables, i have to delete the tables from the document text before chunking (because we are storing tables saparately and it not rquired to store in vector DB)